In [9]:
pip install xgboost


[notice] A new release of pip is available: 24.1.2 -> 26.1.1
[notice] To update, run: C:\Users\neash\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [12]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
import os

def run_clutch_analysis(input_csv, output_csv):
    # 1. LOAD DATA
    if not os.path.exists(input_csv):
        print(f"Error: Could not find {input_csv}. Run the updated data_builder.py first!")
        return
        
    df = pd.read_csv(input_csv)

    # 2. FEATURE ENGINEERING: VENUE NORMALIZATION
    # ---------------------------------------------------------
    # Establishing "Ground Par" to contextualize the target
    match_totals = df[df['innings'] == 1].groupby(['match_id', 'venue'])['current_score'].max().reset_index()
    venue_averages = match_totals.groupby('venue')['current_score'].mean().reset_index()
    venue_averages.columns = ['venue', 'avg_ground_score']
    
    df = df.merge(venue_averages, on='venue', how='left')
    df['target_diff'] = df['target'] - df['avg_ground_score']

    # 3. PREPARE 2ND INNINGS DATA
    # ---------------------------------------------------------
    df_2nd = df[df['innings'] == 2].copy()

    # 4. TRAIN XGBOOST MODEL (THE TEMPORAL FEATURE SET)
    # ---------------------------------------------------------
    # We are now using 10 features: 5 situational + 5 momentum signals
    features = [
        'runs_required', 'wickets_lost', 'balls_left', 
        'current_batter_runs', 'target_diff',
        'rrr', 'last_18_runs', 'last_18_wickets', 
        'pressure_delta', 'strike_balls_rem'
    ]
    
    X = df_2nd[features]
    y = df_2nd['is_winner']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Initialize XGBoost
    model = XGBClassifier(
        n_estimators=100,
        max_depth=6,        # Increased depth to capture more complex momentum interactions
        learning_rate=0.05,  # Slightly lower learning rate for better generalization
        random_state=42,
        eval_metric='logloss'
    )
    
    model.fit(X_train, y_train)
    
    print(f"XGBoost v3.0 (Temporal) Training Complete.")
    print(f"Accuracy Score: {model.score(X_test, y_test):.2%}")
    
    # Check which "Signal" is the strongest
    importances = dict(zip(features, model.feature_importances_))
    print("\nFeature Importance (The Match Pulse):")
    for feat, val in sorted(importances.items(), key=lambda x: x[1], reverse=True):
        print(f" - {feat}: {val:.4f}")

    # 5. PREDICT WIN PROBABILITY (WP)
    # ---------------------------------------------------------
    df_2nd['win_prob'] = model.predict_proba(X)[:, 1]

    # 6. CALCULATE WIN PROBABILITY ADDED (WPA)
    # ---------------------------------------------------------
    df_2nd['prev_win_prob'] = df_2nd.groupby('match_id')['win_prob'].shift(1)
    df_2nd['prev_win_prob'] = df_2nd['prev_win_prob'].fillna(df_2nd['win_prob'])
    df_2nd['wpa'] = df_2nd['win_prob'] - df_2nd['prev_win_prob']

    # 7. CALCULATE LEVERAGE INDEX (LI)
    # ---------------------------------------------------------
    # Standard deviation over a 6-ball window to find high-volatility moments
    df_2nd['li'] = df_2nd.groupby('match_id')['wpa'].transform(lambda x: x.rolling(window=6, min_periods=1).std())
    df_2nd['li'] = df_2nd['li'] / df_2nd['li'].mean()

    # 8. CALCULATE CLUTCH IMPACT
    # ---------------------------------------------------------
    df_2nd['batsman_clutch_impact'] = df_2nd['wpa'] * df_2nd['li']
    df_2nd['bowler_clutch_impact'] = (df_2nd['wpa'] * -1) * df_2nd['li']

    # 9. EXPORT RESULTS
    # ---------------------------------------------------------
    df_2nd.to_csv(output_csv, index=False)
    print(f"\nFinal analytics (v3.0) exported to {output_csv}")

    # 10. GENERATE INSIGHTS
    generate_leaderboards(df_2nd)

def generate_leaderboards(df):
    print("\n" + "="*50)
    print("      CLUTCHLY ANALYTICS SUMMARY (TEMPORAL XGB)")
    print("="*50)

    # Focus on the "Clutch Window" (Last 4 overs)
    death_overs = df[df['balls_left'] <= 24]
    
    print("\n--- TOP 10 DEATH-OVER CLUTCH BATSMEN ---")
    print(death_overs.groupby('batsman')['batsman_clutch_impact'].sum().sort_values(ascending=False).head(10))

    print("\n--- TOP 10 DEATH-OVER CLUTCH BOWLERS ---")
    print(death_overs.groupby('bowler')['bowler_clutch_impact'].sum().sort_values(ascending=False).head(10))

    print("\n--- TOP 10 OVERALL MATCH WINNERS (Total WPA) ---")
    print(df.groupby('batsman')['wpa'].sum().sort_values(ascending=False).head(10))

if __name__ == "__main__":
    input_file = '../data/processed/master_deliveries.csv'
    output_file = '../data/processed/clutch_final_results.csv'
    
    run_clutch_analysis(input_file, output_file)

XGBoost v3.0 (Temporal) Training Complete.
Accuracy Score: 82.06%

Feature Importance (The Match Pulse):
 - rrr: 0.6581
 - pressure_delta: 0.0818
 - runs_required: 0.0507
 - wickets_lost: 0.0501
 - target_diff: 0.0450
 - current_batter_runs: 0.0285
 - last_18_wickets: 0.0274
 - balls_left: 0.0223
 - strike_balls_rem: 0.0192
 - last_18_runs: 0.0169

Final analytics (v3.0) exported to ../data/processed/clutch_final_results.csv

      CLUTCHLY ANALYTICS SUMMARY (TEMPORAL XGB)

--- TOP 10 DEATH-OVER CLUTCH BATSMEN ---
batsman
MS Dhoni          39.131724
RA Jadeja         26.714888
RG Sharma         26.159047
KA Pollard        20.077800
R Tewatia         14.463203
AB de Villiers    13.783797
DJ Bravo          13.136926
DA Miller         11.978301
RK Singh          10.388449
RA Tripathi       10.096835
Name: batsman_clutch_impact, dtype: float64

--- TOP 10 DEATH-OVER CLUTCH BOWLERS ---
bowler
SL Malinga       10.817865
SK Trivedi        5.868076
P Kumar           5.387847
CV Varun          